# Extensions to the Regression Analysis

This notebook complements `03_regression.ipynb`. It does **not** change the primary results. It reads the dataset exported by notebook 03 (`final_regression_dataset.xlsx`, N = 366) and adds the supplementary analyses used to address open methodological points:

1. Wild cluster bootstrap for the joint tests (H1, H3), because only 23 clusters are available.
2. Alternative outcome models for the bounded and concentrated sentiment scores (logit, linear probability, ordered logit).
3. Diagnostics for the Context x Country interaction (cell counts, H3 without PERF, leave-one-company-out, effect coding).
4. Overlap between Information Valence and Country (cross-tabulations and a Valence x Country interaction).
5. Human-AI coding agreement (confusion matrices, changes after re-evaluation). Requires the validation files.
6. Effect sizes (incremental R-squared).
7. Further sensitivity checks: OTHER included, quote/body-text duplicates, Adidas interview indicator, communicator check.

NB: All results are exploratory supplements. Run notebook 03 first so that the exported dataset exists.

## 0. Setup and reproduction check

In [1]:
from pathlib import Path
import re
import warnings
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy
from IPython.display import display

COMPANIES = {
    "adidas": "Germany",
    "puma": "Germany",
    "asics": "Japan",
    "mizuno": "Japan"
}

SENTIMENT_MODELS = ("financial_roberta", "finbert")
MODEL_NAMES = {"financial_roberta": "Financial-RoBERTa", "finbert": "FinBERT"}
VALID_CONTEXTS = ["PERF", "STRAT", "MARKET", "CHALL", "PEOPLE", "STAKE"]
VALID_TEXT_TYPES = ["body_text", "quote", "answer"]

# Locate the processed data folder (same logic as notebook 03)
FINAL_FILE = "final_regression_dataset.xlsx"
folders = [
    Path("../data/processed"),
    Path("data/processed"),
    Path("."),
    Path("/mnt/data")
]

DATA_DIR = next((p for p in folders if (p / FINAL_FILE).exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        f"{FINAL_FILE} not found. Run notebook 03 (section 9) first."
    )

if DATA_DIR.name == "processed" and DATA_DIR.parent.name == "data":
    PROJECT_ROOT = DATA_DIR.resolve().parent.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()

TABLE_DIR = PROJECT_ROOT / "results" / "tables" / "improvements"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

analysis = pd.read_excel(DATA_DIR / FINAL_FILE)

for c in ["context", "valence", "country", "company", "text_type"]:
    analysis[c] = analysis[c].astype(str).str.strip()
analysis["year"] = analysis["year"].astype(int)

if "communication_type" in analysis.columns:
    analysis["communication_type"] = (
        analysis["communication_type"].astype(str).str.strip().str.lower()
    )

print("Data folder:", DATA_DIR.resolve())
print("Results folder:", TABLE_DIR.resolve())
print("Observations:", len(analysis))
print("Clusters:", analysis["cluster_id"].nunique())
assert len(analysis) == 366, "Expected 366 observations"
assert analysis["cluster_id"].nunique() == 23, "Expected 23 clusters"

Data folder: /Users/Saowalak/Desktop/CEOCommunication/data/processed
Results folder: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements
Observations: 366
Clusters: 23


In [2]:
# Formula building blocks (identical to notebook 03)
C_CONTEXT = 'C(context, Treatment(reference="PERF"))'
C_COUNTRY = 'C(country, Treatment(reference="Germany"))'
C_VALENCE = 'C(valence, Treatment(reference="neutral"))'
CONTROLS = f"{C_VALENCE} + C(year) + segment_length"


def f_additive(y, extra=""):
    return f"{y} ~ {C_CONTEXT} + {C_COUNTRY} + {CONTROLS}{extra}"


def f_interaction(y, extra=""):
    return f"{y} ~ {C_CONTEXT} * {C_COUNTRY} + {CONTROLS}{extra}"


def fit(formula, data=analysis, method="cluster", weights=None):
    """OLS (or WLS) with company-year clustered SEs, as in notebook 03."""
    if weights is None:
        reg = smf.ols(formula, data=data)
    else:
        reg = smf.wls(formula, data=data, weights=weights)

    if method == "cluster":
        result = reg.fit(
            cov_type="cluster",
            cov_kwds={"groups": data["cluster_id"]},
            use_t=True
        )
    elif method == "HC3":
        result = reg.fit(cov_type="HC3", use_t=True)
    else:
        raise ValueError(method)

    if result.model.rank != len(result.params):
        raise ValueError(f"Collinearity detected: {formula}")

    return result


def terms_matching(result, *must_contain, exclude_colon=False):
    """Coefficient names containing all given substrings."""
    out = []
    for t in result.params.index:
        if all(s in t for s in must_contain):
            if exclude_colon and ":" in t:
                continue
            out.append(t)
    return out


def joint_f(result, term_names):
    R = np.zeros((len(term_names), len(result.params)))
    for i, name in enumerate(term_names):
        R[i, result.params.index.get_loc(name)] = 1
    test = result.wald_test(R, use_f=True, scalar=True)
    return {
        "F": float(test.statistic),
        "p": float(test.pvalue),
        "df_num": len(term_names),
        "df_den": float(test.df_denom)
    }


def save_table(df, name):
    path = TABLE_DIR / name
    df.to_csv(path, index=False)
    print("Saved:", path)

In [3]:
# Reproduction check against the primary results (thesis Table 1)
repro = []

for m in SENTIMENT_MODELS:
    y = f"{m}_score"
    add = fit(f_additive(y))
    inter = fit(f_interaction(y))

    h1 = joint_f(add, terms_matching(add, "C(context", exclude_colon=True))
    h3 = joint_f(inter, terms_matching(inter, "C(context", "C(country", ":"))
    country_name, = terms_matching(add, "C(country", exclude_colon=True)

    repro.append({"model": m, "test": "H1", "stat": h1["F"], "p": h1["p"]})
    repro.append({"model": m, "test": "H2", "stat": add.params[country_name], "p": add.pvalues[country_name]})
    repro.append({"model": m, "test": "H3", "stat": h3["F"], "p": h3["p"]})

repro = pd.DataFrame(repro)
display(repro.round(4))

print(
    "Expected (thesis Table 1): RoBERTa H1 F=3.335 p=0.0216; H2 b=0.1684 p=0.0015; "
    "H3 F=1.260 p=0.3161 | FinBERT H1 F=12.025 p<0.001; H2 b=0.0903 p=0.0745; "
    "H3 F=5.274 p=0.0025"
)

,model,test,stat,p
0,financial_roberta,H1,3.3352,0.0216
1,financial_roberta,H2,0.1684,0.0015
2,financial_roberta,H3,1.2599,0.3161
3,finbert,H1,12.0249,0.0000
4,finbert,H2,0.0903,0.0745
5,finbert,H3,5.2737,0.0025


Expected (thesis Table 1): RoBERTa H1 F=3.335 p=0.0216; H2 b=0.1684 p=0.0015; H3 F=1.260 p=0.3161 | FinBERT H1 F=12.025 p<0.001; H2 b=0.0903 p=0.0745; H3 F=5.274 p=0.0025


## 1. Wild cluster bootstrap for H1 and H3

With 23 clusters and around 20 parameters in the interaction model, the cluster-robust covariance matrix is imprecise in small samples. As a supplementary check, joint tests are re-evaluated with a **wild cluster restricted bootstrap** (null imposed, Rademacher weights, Wald statistic recomputed with cluster-robust standard errors in every replication). H2 is not bootstrapped because the country coefficient cannot be separated from company membership.

In [4]:
B = 999          # increase to 1999 for the final run
SEED = 20261207
WEIGHTS = "rademacher"   # or "webb"


def wild_cluster_test(y, full_formula, restricted_formula, tested_terms_fn,
                      data=analysis, B=B, seed=SEED, weights=WEIGHTS):

    rng = np.random.default_rng(seed)

    groups = pd.factorize(data["cluster_id"])[0]
    G = groups.max() + 1

    full = smf.ols(full_formula, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": groups},
        use_t=True
    )

    names = list(full.params.index)
    tested = [n for n in names if tested_terms_fn(n)]

    R = np.zeros((len(tested), len(names)))
    for i, n in enumerate(tested):
        R[i, names.index(n)] = 1

    def wald(res):
        return float(res.wald_test(R, use_f=False, scalar=True).statistic)

    observed = wald(full)

    restricted = smf.ols(restricted_formula, data=data).fit()
    fitted_r = restricted.fittedvalues.to_numpy()
    resid_r = restricted.resid.to_numpy()

    X = full.model.exog

    if weights == "rademacher":
        draw = lambda: rng.choice([-1.0, 1.0], size=G)
    else:
        support = np.array([-np.sqrt(1.5), -1, -np.sqrt(0.5),
                            np.sqrt(0.5), 1, np.sqrt(1.5)])
        draw = lambda: rng.choice(support, size=G)

    boot = np.empty(B)

    for b in range(B):
        w = draw()
        y_star = fitted_r + w[groups] * resid_r
        res = sm.OLS(y_star, X).fit(
            cov_type="cluster",
            cov_kwds={"groups": groups},
            use_t=True
        )
        boot[b] = wald(res)

    p_boot = (np.sum(boot >= observed) + 1) / (B + 1)

    return {
        "n_restrictions": len(tested),
        "wald_chi2": observed,
        "p_bootstrap": p_boot
    }


boot_rows = []

for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    # H1: context terms in the additive model
    r1 = wild_cluster_test(
        y,
        f_additive(y),
        f"{y} ~ {C_COUNTRY} + {CONTROLS}",
        lambda n: n.startswith("C(context") and ":" not in n
    )

    # H3: interaction terms
    r3 = wild_cluster_test(
        y,
        f_interaction(y),
        f_additive(y),
        lambda n: ":" in n and "C(context" in n
    )

    add = fit(f_additive(y))
    inter = fit(f_interaction(y))

    boot_rows.append({
        "model": MODEL_NAMES[m], "test": "H1",
        "p_cluster_t": joint_f(add, terms_matching(add, "C(context", exclude_colon=True))["p"],
        **r1
    })
    boot_rows.append({
        "model": MODEL_NAMES[m], "test": "H3",
        "p_cluster_t": joint_f(inter, terms_matching(inter, "C(context", "C(country", ":"))["p"],
        **r3
    })

wild_bootstrap = pd.DataFrame(boot_rows)
display(wild_bootstrap.round(4))
save_table(wild_bootstrap, "01_wild_cluster_bootstrap.csv")

,model,test,p_cluster_t,n_restrictions,wald_chi2,p_bootstrap
0,Financial-RoBERTa,H1,0.0216,5,16.6758,0.071
1,Financial-RoBERTa,H3,0.3161,5,6.2993,0.569
2,FinBERT,H1,0.0000,5,60.1246,0.004
3,FinBERT,H3,0.0025,5,26.3687,0.047


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/01_wild_cluster_bootstrap.csv


## 2. Alternative outcome models

Financial-RoBERTa assigns most segments a score at or near +1, so the continuous score behaves almost like a binary outcome. Three supplementary specifications are estimated:

- **Linear probability model (LPM):** indicator for the predicted class being *positive*, with clustered standard errors.
- **Logit:** the same indicator, logistic link. If the model fails because of (quasi-)separation (for example when negative valence never receives a positive prediction), the LPM remains the reference.
- **Ordered logit:** the three predicted classes (negative < neutral < positive).

In [5]:
for m in SENTIMENT_MODELS:

    probs = np.column_stack([
        analysis[f"{m}_positive_prob"],
        analysis[f"{m}_neutral_prob"],
        analysis[f"{m}_negative_prob"]
    ])

    cls = probs.argmax(axis=1)           # 0 = positive, 1 = neutral, 2 = negative

    analysis[f"{m}_pred_positive"] = (cls == 0).astype(int)
    analysis[f"{m}_pred_class"] = pd.Categorical(
        pd.Series(cls).map({0: "positive", 1: "neutral", 2: "negative"}),
        categories=["negative", "neutral", "positive"],
        ordered=True
    )

print("Predicted class shares:")
for m in SENTIMENT_MODELS:
    print(
        MODEL_NAMES[m],
        analysis[f"{m}_pred_class"].value_counts(normalize=True).round(3).to_dict()
    )

print("\nShare predicted positive, by country and context:")
for m in SENTIMENT_MODELS:
    print("\n", MODEL_NAMES[m])
    display(
        analysis.pivot_table(
            index="country", columns="context",
            values=f"{m}_pred_positive", aggfunc="mean"
        )[VALID_CONTEXTS].round(2)
    )

Predicted class shares:
Financial-RoBERTa {'positive': 0.724, 'neutral': 0.21, 'negative': 0.066}
FinBERT {'positive': 0.538, 'neutral': 0.41, 'negative': 0.052}

Share predicted positive, by country and context:

 Financial-RoBERTa


context,PERF,STRAT,MARKET,CHALL,PEOPLE,STAKE
country,,,,,,
Germany,0.83,0.65,0.89,0.33,0.61,0.60
Japan,0.87,0.80,0.75,0.47,0.80,0.78



 FinBERT


context,PERF,STRAT,MARKET,CHALL,PEOPLE,STAKE
country,,,,,,
Germany,0.62,0.53,0.66,0.33,0.58,0.50
Japan,0.91,0.59,0.50,0.42,0.30,0.34


In [6]:
alt_rows = []
alt_models = {}

for m in SENTIMENT_MODELS:

    y = f"{m}_pred_positive"

    # (a) Linear probability model
    lpm_add = fit(f_additive(y), data=analysis)
    lpm_int = fit(f_interaction(y), data=analysis)

    alt_rows += [
        {"model": MODEL_NAMES[m], "estimator": "LPM", "test": "H1",
         **joint_f(lpm_add, terms_matching(lpm_add, "C(context", exclude_colon=True))},
        {"model": MODEL_NAMES[m], "estimator": "LPM", "test": "H3",
         **joint_f(lpm_int, terms_matching(lpm_int, "C(context", "C(country", ":"))}
    ]

    cname, = terms_matching(lpm_add, "C(country", exclude_colon=True)
    alt_rows.append({
        "model": MODEL_NAMES[m], "estimator": "LPM", "test": "H2 (coef.)",
        "coefficient": lpm_add.params[cname], "p": lpm_add.pvalues[cname]
    })

    # (b) Logit with clustered SEs
    try:
        lg_add = smf.logit(f_additive(y), data=analysis).fit(
            disp=0,
            cov_type="cluster",
            cov_kwds={"groups": pd.factorize(analysis["cluster_id"])[0]}
        )
        lg_int = smf.logit(f_interaction(y), data=analysis).fit(
            disp=0,
            cov_type="cluster",
            cov_kwds={"groups": pd.factorize(analysis["cluster_id"])[0]}
        )

        def wald_chi2(res, names):
            R = np.zeros((len(names), len(res.params)))
            for i, n in enumerate(names):
                R[i, res.params.index.get_loc(n)] = 1
            t = res.wald_test(R, use_f=False, scalar=True)
            return {"F": float(t.statistic), "p": float(t.pvalue), "df_num": len(names)}

        alt_rows += [
            {"model": MODEL_NAMES[m], "estimator": "Logit (Wald chi2)", "test": "H1",
             **wald_chi2(lg_add, terms_matching(lg_add, "C(context", exclude_colon=True))},
            {"model": MODEL_NAMES[m], "estimator": "Logit (Wald chi2)", "test": "H3",
             **wald_chi2(lg_int, terms_matching(lg_int, "C(context", "C(country", ":"))}
        ]

        cname, = terms_matching(lg_add, "C(country", exclude_colon=True)
        alt_rows.append({
            "model": MODEL_NAMES[m], "estimator": "Logit", "test": "H2 (coef.)",
            "coefficient": lg_add.params[cname], "p": lg_add.pvalues[cname]
        })
        alt_models[(m, "logit_add")] = lg_add

    except Exception as e:
        print(f"Logit failed for {MODEL_NAMES[m]} (possible separation): {type(e).__name__}: {e}")

alt_results = pd.DataFrame(alt_rows)
display(alt_results.round(4))
save_table(alt_results, "02_alternative_outcome_models.csv")

,model,estimator,test,F,p,df_num,df_den,coefficient
0,Financial-RoBERTa,LPM,H1,2.8213,0.0408,5.0,22.0,NaN
1,Financial-RoBERTa,LPM,H3,1.2595,0.3162,5.0,22.0,NaN
2,Financial-RoBERTa,LPM,H2 (coef.),NaN,0.0578,NaN,NaN,0.0980
3,Financial-RoBERTa,Logit (Wald chi2),H1,13.5985,0.0184,5.0,NaN,NaN
4,Financial-RoBERTa,Logit (Wald chi2),H3,7.4939,0.1864,5.0,NaN,NaN
5,Financial-RoBERTa,Logit,H2 (coef.),NaN,0.0487,NaN,NaN,0.6743
6,FinBERT,LPM,H1,6.1708,0.0010,5.0,22.0,NaN
7,FinBERT,LPM,H3,2.8185,0.0409,5.0,22.0,NaN
8,FinBERT,LPM,H2 (coef.),NaN,0.8722,NaN,NaN,0.0123
9,FinBERT,Logit (Wald chi2),H1,29.9089,0.0000,5.0,NaN,NaN


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/02_alternative_outcome_models.csv


In [7]:
# Ordered logit on the three predicted classes (additive specification)
from statsmodels.miscmodels.ordinal_model import OrderedModel

ord_rows = []

for m in SENTIMENT_MODELS:

    rhs = f"{C_CONTEXT} + {C_COUNTRY} + {CONTROLS}"

    X = patsy.dmatrix(rhs, analysis, return_type="dataframe")
    X = X.drop(columns="Intercept")

    mod = OrderedModel(
        analysis[f"{m}_pred_class"],
        X,
        distr="logit"
    )

    try:
        res = mod.fit(
            method="bfgs",
            disp=False,
            maxiter=2000,
            cov_type="cluster",
            cov_kwds={"groups": pd.factorize(analysis["cluster_id"])[0]}
        )
        se_note = "clustered SEs"
    except Exception as e:
        print(f"Clustered SEs unavailable for ordered logit ({type(e).__name__}); using default SEs.")
        res = mod.fit(method="bfgs", disp=False, maxiter=2000)
        se_note = "default SEs (not clustered)"

    for term in X.columns:
        if "C(context" in term or "C(country" in term or "C(valence" in term:
            ord_rows.append({
                "model": MODEL_NAMES[m],
                "SE": se_note,
                "term": term,
                "coefficient": res.params[term],
                "p": res.pvalues[term]
            })

ordered_results = pd.DataFrame(ord_rows)
display(ordered_results.round(4))
save_table(ordered_results, "03_ordered_logit.csv")

,model,SE,term,coefficient,p
0,Financial-RoBERTa,clustered SEs,"C(context, Treatment(reference=""PERF""))[T.CHALL]",-1.9032,0.0052
1,Financial-RoBERTa,clustered SEs,"C(context, Treatment(reference=""PERF""))[T.MARKET]",-0.3785,0.4124
2,Financial-RoBERTa,clustered SEs,"C(context, Treatment(reference=""PERF""))[T.PEOPLE]",-1.3426,0.0188
3,Financial-RoBERTa,clustered SEs,"C(context, Treatment(reference=""PERF""))[T.STAKE]",-1.3997,0.0237
4,Financial-RoBERTa,clustered SEs,"C(context, Treatment(reference=""PERF""))[T.STRAT]",-0.9922,0.0961
5,Financial-RoBERTa,clustered SEs,"C(country, Treatment(reference=""Germany""))[T.J...",0.9025,0.0031
6,Financial-RoBERTa,clustered SEs,"C(valence, Treatment(reference=""neutral""))[T.n...",-4.2095,0.0000
7,Financial-RoBERTa,clustered SEs,"C(valence, Treatment(reference=""neutral""))[T.p...",0.6875,0.0111
8,FinBERT,clustered SEs,"C(context, Treatment(reference=""PERF""))[T.CHALL]",-1.0218,0.0246
9,FinBERT,clustered SEs,"C(context, Treatment(reference=""PERF""))[T.MARKET]",-0.4084,0.2414


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/03_ordered_logit.csv


## 3. Diagnostics for the Context x Country interaction

The FinBERT interaction is expressed relative to PERF. The Japanese PERF cell is small and, in the initial data, Mizuno contributes very few PERF segments, so it is essentially an Asics cell. The following checks show how far the interaction result depends on that cell.

In [8]:
# 3.1 Cell counts and cell means by company

print("Segments by company and context:")
display(pd.crosstab(analysis["company"], analysis["context"], margins=True)[VALID_CONTEXTS + ["All"]])

cell_tables = []
for m in SENTIMENT_MODELS:
    t = analysis.pivot_table(
        index="company", columns="context",
        values=f"{m}_score", aggfunc="mean"
    )[VALID_CONTEXTS].round(3)
    print("\nMean score by company and context:", MODEL_NAMES[m])
    display(t)
    t = t.reset_index()
    t.insert(0, "model", MODEL_NAMES[m])
    cell_tables.append(t)

save_table(pd.concat(cell_tables), "03_cell_means_company_context.csv")

Segments by company and context:


context,PERF,STRAT,MARKET,CHALL,PEOPLE,STAKE,All
company,,,,,,,
adidas,34,28,14,13,18,7,114
asics,21,41,12,8,15,28,125
mizuno,2,8,4,11,5,13,43
puma,19,15,21,11,15,3,84
All,76,92,51,43,53,51,366



Mean score by company and context: Financial-RoBERTa


context,PERF,STRAT,MARKET,CHALL,PEOPLE,STAKE
company,,,,,,
adidas,0.871,0.604,0.842,-0.160,0.602,0.635
asics,0.808,0.787,0.652,0.632,0.778,0.749
mizuno,0.976,0.786,0.750,0.331,0.794,0.669
puma,0.423,0.696,0.904,-0.092,0.604,0.640



Mean score by company and context: FinBERT


context,PERF,STRAT,MARKET,CHALL,PEOPLE,STAKE
company,,,,,,
adidas,0.474,0.402,0.400,0.037,0.334,0.390
asics,0.797,0.555,0.346,0.380,0.404,0.331
mizuno,0.430,0.542,0.615,0.360,0.177,0.470
puma,0.491,0.690,0.747,-0.190,0.690,0.614


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/03_cell_means_company_context.csv


In [9]:
# 3.2 H3 without the PERF cell (reference category: MARKET)

h3_rows = []

no_perf = analysis[analysis["context"] != "PERF"].copy()

for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    formula = (
        f'{y} ~ C(context, Treatment(reference="MARKET")) * {C_COUNTRY} + {CONTROLS}'
    )

    res = fit(formula, data=no_perf)

    h3_rows.append({
        "model": MODEL_NAMES[m],
        "sample": "Without PERF",
        "N": int(res.nobs),
        **joint_f(res, terms_matching(res, "C(context", "C(country", ":"))
    })

    # Full-sample reference
    full = fit(f_interaction(y))
    h3_rows.append({
        "model": MODEL_NAMES[m],
        "sample": "Full sample",
        "N": int(full.nobs),
        **joint_f(full, terms_matching(full, "C(context", "C(country", ":"))
    })

# 3.3 Leave-one-company-out
for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    for company in COMPANIES:

        sub = analysis[analysis["company"] != company].copy()

        try:
            res = fit(f_interaction(y), data=sub)
            h3_rows.append({
                "model": MODEL_NAMES[m],
                "sample": f"Excluding {company}",
                "N": int(res.nobs),
                **joint_f(res, terms_matching(res, "C(context", "C(country", ":"))
            })
        except Exception as e:
            print(f"{MODEL_NAMES[m]}, excluding {company}: {type(e).__name__}: {e}")

h3_diagnostics = pd.DataFrame(h3_rows)
display(h3_diagnostics.round(4))
save_table(h3_diagnostics, "04_h3_diagnostics.csv")

,model,sample,N,F,p,df_num,df_den
0,Financial-RoBERTa,Without PERF,290,1.7630,0.1723,4,22.0
1,Financial-RoBERTa,Full sample,366,1.2599,0.3161,5,22.0
2,FinBERT,Without PERF,290,3.5574,0.0220,4,22.0
3,FinBERT,Full sample,366,5.2737,0.0025,5,22.0
4,Financial-RoBERTa,Excluding adidas,252,2.2348,0.1011,5,16.0
5,Financial-RoBERTa,Excluding puma,282,1.8121,0.1673,5,16.0
6,Financial-RoBERTa,Excluding asics,241,0.4591,0.8008,5,16.0
7,Financial-RoBERTa,Excluding mizuno,323,8.0280,0.0005,5,17.0
8,FinBERT,Excluding adidas,252,8.3335,0.0005,5,16.0
9,FinBERT,Excluding puma,282,2.3595,0.0874,5,16.0


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/04_h3_diagnostics.csv


In [10]:
# 3.4 Effect (sum-to-zero) coding: interaction terms are deviations from the grand mean
# rather than contrasts against PERF.

for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    formula = (
        f"{y} ~ C(context, Sum) * C(country, Sum) + {C_VALENCE} + C(year) + segment_length"
    )

    res = fit(formula)

    print("\n", MODEL_NAMES[m], "- effect-coded interaction model")

    coef = pd.DataFrame({
        "coefficient": res.params,
        "SE": res.bse,
        "p": res.pvalues
    })

    display(coef.loc[[i for i in coef.index if "C(context" in i or "C(country" in i]].round(3))

    inter_terms = terms_matching(res, "C(context", "C(country", ":")
    print("Joint interaction test:", {k: round(v, 4) for k, v in joint_f(res, inter_terms).items()})

print("\nNote: with Sum coding, the omitted level (alphabetically last) is the negative sum of the others.")


 Financial-RoBERTa - effect-coded interaction model


,coefficient,SE,p
"C(context, Sum)[S.CHALL]",-0.154,0.088,0.094
"C(context, Sum)[S.MARKET]",0.083,0.052,0.126
"C(context, Sum)[S.PEOPLE]",-0.039,0.053,0.472
"C(context, Sum)[S.PERF]",0.123,0.053,0.031
"C(context, Sum)[S.STAKE]",-0.062,0.071,0.394
"C(country, Sum)[S.Germany]",-0.088,0.025,0.002
"C(context, Sum)[S.CHALL]:C(country, Sum)[S.Germany]",-0.113,0.093,0.240
"C(context, Sum)[S.MARKET]:C(country, Sum)[S.Germany]",0.115,0.049,0.027
"C(context, Sum)[S.PEOPLE]:C(country, Sum)[S.Germany]",-0.020,0.052,0.706
"C(context, Sum)[S.PERF]:C(country, Sum)[S.Germany]",0.047,0.049,0.345


Joint interaction test: {'F': 1.2599, 'p': 0.3161, 'df_num': 5, 'df_den': 22.0}

 FinBERT - effect-coded interaction model


,coefficient,SE,p
"C(context, Sum)[S.CHALL]",-0.085,0.068,0.225
"C(context, Sum)[S.MARKET]",0.037,0.031,0.240
"C(context, Sum)[S.PEOPLE]",-0.092,0.040,0.031
"C(context, Sum)[S.PERF]",0.169,0.031,0.000
"C(context, Sum)[S.STAKE]",-0.093,0.046,0.057
"C(country, Sum)[S.Germany]",-0.038,0.020,0.078
"C(context, Sum)[S.CHALL]:C(country, Sum)[S.Germany]",-0.112,0.078,0.165
"C(context, Sum)[S.MARKET]:C(country, Sum)[S.Germany]",0.091,0.034,0.015
"C(context, Sum)[S.PEOPLE]:C(country, Sum)[S.Germany]",0.079,0.036,0.038
"C(context, Sum)[S.PERF]:C(country, Sum)[S.Germany]",-0.094,0.033,0.009


Joint interaction test: {'F': 5.2737, 'p': 0.0025, 'df_num': 5, 'df_den': 22.0}

Note: with Sum coding, the omitted level (alphabetically last) is the negative sum of the others.


## 4. Overlap between Information Valence, Context and Country

The valence coefficients are identified from the segments that actually vary in valence. If negative valence is almost absent in Japanese documents, the valence slope is learned mainly from German text. The cross-tabulations below document this, and a Valence x Country interaction tests whether one common slope is reasonable.

In [11]:
print("Valence by country and context:")
vc = pd.crosstab([analysis["country"], analysis["context"]], analysis["valence"], margins=True)
display(vc)

print("\nValence by company:")
display(pd.crosstab(analysis["company"], analysis["valence"], margins=True))

print("\nRow shares of valence within country (%):")
display((pd.crosstab(analysis["country"], analysis["valence"], normalize="index") * 100).round(1))

save_table(vc.reset_index(), "05_valence_context_country.csv")

Valence by country and context:


valence          negative  neutral  positive  All
country context                                  
Germany CHALL          10        4        10   24
        MARKET          0        6        29   35
        PEOPLE          0        6        27   33
        PERF            5        7        41   53
        STAKE           0        2         8   10
        STRAT           0       26        17   43
Japan   CHALL           4       12         3   19
        MARKET          1        9         6   16
        PEOPLE          0        9        11   20
        PERF            2        4        17   23
        STAKE           0       27        14   41
        STRAT           1       40         8   49
All                    23      152       191  366


Valence by company:


valence,negative,neutral,positive,All
company,,,,
adidas,5,28,81,114
asics,8,71,46,125
mizuno,0,30,13,43
puma,10,23,51,84
All,23,152,191,366



Row shares of valence within country (%):


valence,negative,neutral,positive
country,,,
Germany,7.6,25.8,66.7
Japan,4.8,60.1,35.1


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/05_valence_context_country.csv


In [12]:
val_rows = []

for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    formula = (
        f"{y} ~ {C_CONTEXT} + {C_COUNTRY} * {C_VALENCE} + C(year) + segment_length"
    )

    try:
        res = fit(formula)

        inter_terms = terms_matching(res, "C(country", "C(valence", ":")

        val_rows.append({
            "model": MODEL_NAMES[m],
            "test": "Valence x Country",
            **joint_f(res, inter_terms)
        })

        coef = pd.DataFrame({"coefficient": res.params, "p": res.pvalues}).loc[inter_terms]
        print("\n", MODEL_NAMES[m])
        display(coef.round(3))

    except Exception as e:
        print(f"{MODEL_NAMES[m]}: {type(e).__name__}: {e}")

valence_country = pd.DataFrame(val_rows)
display(valence_country.round(4))
save_table(valence_country, "06_valence_country_interaction.csv")


 Financial-RoBERTa


,coefficient,p
"C(country, Treatment(reference=""Germany""))[T.Japan]:C(valence, Treatment(reference=""neutral""))[T.negative]",0.507,0.002
"C(country, Treatment(reference=""Germany""))[T.Japan]:C(valence, Treatment(reference=""neutral""))[T.positive]",-0.133,0.112



 FinBERT


,coefficient,p
"C(country, Treatment(reference=""Germany""))[T.Japan]:C(valence, Treatment(reference=""neutral""))[T.negative]",0.774,0.002
"C(country, Treatment(reference=""Germany""))[T.Japan]:C(valence, Treatment(reference=""neutral""))[T.positive]",-0.039,0.642


,model,test,F,p,df_num,df_den
0,Financial-RoBERTa,Valence x Country,9.4185,0.0011,2,22.0
1,FinBERT,Valence x Country,7.4035,0.0035,2,22.0


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/06_valence_country_interaction.csv


## 5. Effect sizes: incremental R-squared

Statistical significance of the joint tests does not show how much of the variance Leadership Context explains beyond Information Valence. Incremental R-squared is computed by dropping one block at a time from the additive model (and the interaction block from the interaction model).

In [13]:
r2_rows = []

for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    full = smf.ols(f_additive(y), data=analysis).fit()
    no_ctx = smf.ols(f"{y} ~ {C_COUNTRY} + {CONTROLS}", data=analysis).fit()
    no_val = smf.ols(f"{y} ~ {C_CONTEXT} + {C_COUNTRY} + C(year) + segment_length", data=analysis).fit()
    no_cty = smf.ols(f"{y} ~ {C_CONTEXT} + {CONTROLS}", data=analysis).fit()
    no_len = smf.ols(f"{y} ~ {C_CONTEXT} + {C_COUNTRY} + {C_VALENCE} + C(year)", data=analysis).fit()
    inter = smf.ols(f_interaction(y), data=analysis).fit()
    val_only = smf.ols(f"{y} ~ {C_VALENCE}", data=analysis).fit()

    r2_rows.append({
        "model": MODEL_NAMES[m],
        "R2 additive": full.rsquared,
        "R2 valence only": val_only.rsquared,
        "dR2 context": full.rsquared - no_ctx.rsquared,
        "dR2 valence": full.rsquared - no_val.rsquared,
        "dR2 country": full.rsquared - no_cty.rsquared,
        "dR2 segment length": full.rsquared - no_len.rsquared,
        "dR2 context x country": inter.rsquared - full.rsquared,
        "SD of outcome": analysis[y].std()
    })

effect_sizes = pd.DataFrame(r2_rows)
display(effect_sizes.round(3))
save_table(effect_sizes, "07_effect_sizes.csv")

print("Note: block R-squared contributions are not additive when regressors are correlated.")

,model,R2 additive,R2 valence only,dR2 context,dR2 valence,dR2 country,dR2 segment length,dR2 context x country,SD of outcome
0,Financial-RoBERTa,0.392,0.335,0.026,0.238,0.017,0.003,0.010,0.577
1,FinBERT,0.352,0.270,0.048,0.215,0.009,0.020,0.027,0.422


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/07_effect_sizes.csv
Note: block R-squared contributions are not additive when regressors are correlated.


## 7. Further sensitivity checks

### 7.1 OTHER segments included

The residual category OTHER is excluded from the primary sample. It is mostly neutral and, in the initial data, more frequent in some companies than in others. The additive and interaction models are re-estimated with OTHER as a seventh category (reference remains PERF), using the same text-type restriction.

In [18]:
# Rebuild the pre-exclusion dataset (all contexts incl. OTHER) from the company files

import re

def norm_cols(d):
    d = d.copy()
    d.columns = [
        re.sub(r"[^a-z0-9]+", "_", str(c).lower()).strip("_")
        for c in d.columns
    ]
    return d

def load_company(company):
    d = norm_cols(pd.read_excel(DATA_DIR / f"{company}_sentiment.xlsx"))
    d = d.rename(columns={
        "leadership_context": "context",
        "information_valence": "valence",
        "information_value": "valence"
    })
    d["company"] = company
    d["country"] = COMPANIES[company]
    return d

raw_all = pd.concat([load_company(c) for c in COMPANIES], ignore_index=True)
raw_all.head()

,id,text,text_type,context,valence,communication_type,source_type,company,country,year,...,financial_roberta_sentiment,financial_roberta_positive_prob,financial_roberta_neutral_prob,financial_roberta_negative_prob,financial_roberta_n_chunks,finbert_sentiment,finbert_positive_prob,finbert_neutral_prob,finbert_negative_prob,finbert_n_chunks
0,DEU20A00-00,2020 was a year like no other. Despite all the...,quote,CHALL,positive,letter,annual_report,adidas,Germany,2020,...,positive,0.997946,0.001545,0.000509,1,positive,0.701935,0.286729,0.011336,1
1,DEU20A01-00,"DEAR SHAREHOLDERS,",title,STAKE,positive,letter,annual_report,adidas,Germany,2020,...,neutral,0.000934,0.998457,0.000609,1,neutral,0.050315,0.556621,0.393064,1
2,DEU20A01-01,2020 was a year like no other. When we started...,body_text,PERF,positive,letter,annual_report,adidas,Germany,2020,...,positive,0.998254,0.001086,0.000660,1,positive,0.774036,0.215756,0.010208,1
3,DEU20A01-02,Then covid-19 struck the world with full force...,body_text,CHALL,negative,letter,annual_report,adidas,Germany,2020,...,negative,0.000265,0.000669,0.999066,1,negative,0.015411,0.069427,0.915162,1
4,DEU20A01-03,Covid-19 first hit adidas in late January in C...,body_text,CHALL,neutral,letter,annual_report,adidas,Germany,2020,...,negative,0.000227,0.001632,0.998140,1,negative,0.039629,0.069057,0.891314,1


In [20]:
for c in ["context", "valence", "text_type"]:
    raw_all[c] = raw_all[c].astype(str).str.strip()
    raw_all["context"] = raw_all["context"].str.upper()
    raw_all["valence"] = raw_all["valence"].str.lower()
    raw_all["text_type"] = raw_all["text_type"].str.lower()
    raw_all["year"] = pd.to_numeric(raw_all["year"], errors="coerce")

for m in SENTIMENT_MODELS:
    for v in ("positive", "neutral", "negative"):
        raw_all[f"{m}_{v}_prob"] = pd.to_numeric(raw_all[f"{m}_{v}_prob"], errors="coerce")
    raw_all[f"{m}_score"] = raw_all[f"{m}_positive_prob"] - raw_all[f"{m}_negative_prob"]

with_other = raw_all.loc[
    raw_all["context"].isin(VALID_CONTEXTS + ["OTHER"])
    & raw_all["text_type"].isin(VALID_TEXT_TYPES)
    & raw_all["text"].notna()
    & raw_all["text"].astype(str).str.strip().ne("")
].dropna(subset=["year"] + [f"{m}_score" for m in SENTIMENT_MODELS]).copy()

with_other["year"] = with_other["year"].astype(int)
with_other["segment_length"] = with_other["text"].astype(str).str.split().str.len()
with_other["cluster_id"] = with_other["company"] + "_" + with_other["year"].astype(str)

print("Observations including OTHER:", len(with_other))
print("OTHER observations:", (with_other["context"] == "OTHER").sum())
assert len(with_other) - (with_other["context"] == "OTHER").sum() == 366

other_rows = []

for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    add = fit(f_additive(y), data=with_other)
    inter = fit(f_interaction(y), data=with_other)

    other_rows += [
        {"model": MODEL_NAMES[m], "test": "H1 (7 categories)",
         **joint_f(add, terms_matching(add, "C(context", exclude_colon=True))},
        {"model": MODEL_NAMES[m], "test": "H3 (7 categories)",
         **joint_f(inter, terms_matching(inter, "C(context", "C(country", ":"))}
    ]

    cname, = terms_matching(add, "C(country", exclude_colon=True)
    other_rows.append({
        "model": MODEL_NAMES[m], "test": "H2 coefficient (7 categories)",
        "coefficient": add.params[cname], "p": add.pvalues[cname]
    })

    oname = [t for t in add.params.index if "[T.OTHER]" in t][0]
    other_rows.append({
        "model": MODEL_NAMES[m], "test": "OTHER vs PERF",
        "coefficient": add.params[oname], "p": add.pvalues[oname]
    })

other_included = pd.DataFrame(other_rows)
display(other_included.round(4))
save_table(other_included, "08_other_included.csv")

Observations including OTHER: 411
OTHER observations: 45


,model,test,F,p,df_num,df_den,coefficient
0,Financial-RoBERTa,H1 (7 categories),4.0471,0.0070,6.0,22.0,NaN
1,Financial-RoBERTa,H3 (7 categories),1.6723,0.1751,6.0,22.0,NaN
2,Financial-RoBERTa,H2 coefficient (7 categories),NaN,0.0050,NaN,NaN,0.1496
3,Financial-RoBERTa,OTHER vs PERF,NaN,0.0034,NaN,NaN,-0.2611
4,FinBERT,H1 (7 categories),16.0367,0.0000,6.0,22.0,NaN
5,FinBERT,H3 (7 categories),12.7056,0.0000,6.0,22.0,NaN
6,FinBERT,H2 coefficient (7 categories),NaN,0.1486,NaN,NaN,0.0734
7,FinBERT,OTHER vs PERF,NaN,0.0000,NaN,NaN,-0.3720


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/08_other_included.csv


### 7.2 Adidas interview indicator

Adidas statements are letters in 2020 to 2021 and interviews from 2022. This format shift is identified within Germany and can be added without creating perfect collinearity with Country. It is a control for format, not a general Communication Type variable.

In [24]:
analysis["adidas_interview"] = (
    (analysis["company"] == "adidas")
    & (analysis["communication_type"] == "interview")
).astype(int)

print(pd.crosstab([analysis["company"], analysis["year"]], analysis["communication_type"]))

fmt_rows = []

for m in SENTIMENT_MODELS:

    y = f"{m}_score"

    try:
        add = fit(f_additive(y, extra=" + adidas_interview"))
        inter = fit(f_interaction(y, extra=" + adidas_interview"))

        cname, = terms_matching(add, "C(country", exclude_colon=True)

        fmt_rows += [
            {"model": MODEL_NAMES[m], "test": "H1",
             **joint_f(add, terms_matching(add, "C(context", exclude_colon=True))},
            {"model": MODEL_NAMES[m], "test": "H2 (coef.)",
             "coefficient": add.params[cname], "p": add.pvalues[cname]},
            {"model": MODEL_NAMES[m], "test": "H3",
             **joint_f(inter, terms_matching(inter, "C(context", "C(country", ":"))},
            {"model": MODEL_NAMES[m], "test": "adidas_interview (coef.)",
             "coefficient": add.params["adidas_interview"], "p": add.pvalues["adidas_interview"]}
        ]
    except Exception as e:
        print(f"{MODEL_NAMES[m]}: {type(e).__name__}: {e}")

adidas_format = pd.DataFrame(fmt_rows)
display(adidas_format.round(4))
save_table(adidas_format, "09_adidas_interview_control.csv")

communication_type  interview  letter  message
company year                                  
adidas  2020                0      20        0
        2021                0      14        0
        2022               26       0        0
        2023               18       0        0
        2024               20       0        0
        2025               16       0        0
asics   2020                0       0       37
        2021                0       0       18
        2022                0       0       20
        2023                0       0       21
        2024                0       0       14
        2025                0       0       15
mizuno  2021                0       0        9
        2022                0       0        7
        2023                0       0        7
        2024                0       0        7
        2025                0       0       13
puma    2020                0      14        0
        2021                0      13        0
        2022 

,model,test,F,p,df_num,df_den,coefficient
0,Financial-RoBERTa,H1,4.0015,0.0098,5.0,22.0,NaN
1,Financial-RoBERTa,H2 (coef.),NaN,0.0031,NaN,NaN,0.1230
2,Financial-RoBERTa,H3,1.1740,0.3534,5.0,22.0,NaN
3,Financial-RoBERTa,adidas_interview (coef.),NaN,0.1192,NaN,NaN,-0.1214
4,FinBERT,H1,20.2575,0.0000,5.0,22.0,NaN
5,FinBERT,H2 (coef.),NaN,0.5613,NaN,NaN,-0.0274
6,FinBERT,H3,4.9218,0.0036,5.0,22.0,NaN
7,FinBERT,adidas_interview (coef.),NaN,0.0000,NaN,NaN,-0.3142


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/09_adidas_interview_control.csv


### 7.3 Communicators and CEO changes

Changes of the person signing the statement provide variation that is not company-level. If Bjørn Gulden signs Puma statements until his move and Adidas statements afterwards, a person indicator can be added. Check the spelling in `communicator_name` before using the indicator.

In [25]:
if "communicator_name" in analysis.columns:

    analysis["communicator_name"] = analysis["communicator_name"].astype(str).str.strip()

    who = (
        analysis.groupby(["company", "communicator_name", "year"])
        .agg(
            n=("segment_id", "size"),
            roberta_mean=("financial_roberta_score", "mean"),
            finbert_mean=("finbert_score", "mean")
        )
        .round(3)
        .reset_index()
    )

    display(who)
    save_table(who, "10_communicators.csv")

    # ---- adapt the pattern if needed ----
    PERSON_PATTERN = "gulden"
    # --------------------------------------

    analysis["same_person_gulden"] = (
        analysis["communicator_name"].str.contains(PERSON_PATTERN, case=False, na=False)
    ).astype(int)

    print("Segments signed by the selected person:", analysis["same_person_gulden"].sum())
    display(
        analysis[analysis["same_person_gulden"] == 1]
        .groupby(["company", "year"]).size().rename("n").to_frame()
    )

    person_rows = []

    for m in SENTIMENT_MODELS:

        y = f"{m}_score"

        try:
            add = fit(f_additive(y, extra=" + same_person_gulden"))
            cname, = terms_matching(add, "C(country", exclude_colon=True)

            person_rows += [
                {"model": MODEL_NAMES[m], "term": "same person (coef.)",
                 "coefficient": add.params["same_person_gulden"],
                 "p": add.pvalues["same_person_gulden"]},
                {"model": MODEL_NAMES[m], "term": "Japan (coef.)",
                 "coefficient": add.params[cname], "p": add.pvalues[cname]},
                {"model": MODEL_NAMES[m], "term": "H1 joint test",
                 "coefficient": np.nan,
                 "p": joint_f(add, terms_matching(add, "C(context", exclude_colon=True))["p"]}
            ]
        except Exception as e:
            print(f"{MODEL_NAMES[m]}: {type(e).__name__}: {e}")

    person_results = pd.DataFrame(person_rows)
    display(person_results.round(4))
    save_table(person_results, "11_person_indicator.csv")

else:
    print("Column communicator_name not found in the exported dataset.")

,company,communicator_name,year,n,roberta_mean,finbert_mean
0,adidas,Bjørn Gulden,2022,26,0.245,0.159
1,adidas,Bjørn Gulden,2023,18,0.889,0.460
2,adidas,Bjørn Gulden,2024,20,0.612,0.327
3,adidas,Bjørn Gulden,2025,16,0.756,0.390
4,adidas,Kasper Rorsted,2020,20,0.640,0.424
5,adidas,Kasper Rorsted,2021,14,0.859,0.607
6,asics,Mitsuyuki Tominaga,2023,21,0.794,0.519
7,asics,Motoi Oyama,2020,37,0.582,0.381
8,asics,Yasuhito Hirota,2021,18,0.834,0.590
9,asics,Yasuhito Hirota,2022,20,0.856,0.491


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/10_communicators.csv
Segments signed by the selected person: 107


n
company year    
adidas  2022  26
        2023  18
        2024  20
        2025  16
puma    2020  14
        2021  13

,model,term,coefficient,p
0,Financial-RoBERTa,same person (coef.),-0.0917,0.1547
1,Financial-RoBERTa,Japan (coef.),0.1195,0.0079
2,Financial-RoBERTa,H1 joint test,NaN,0.0159
3,FinBERT,same person (coef.),-0.2667,0.0005
4,FinBERT,Japan (coef.),-0.0521,0.3210
5,FinBERT,H1 joint test,NaN,0.0000


Saved: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements/11_person_indicator.csv


## 8. Summary of supplementary results

The table below collects the main p-values for quick comparison with the primary results in notebook 03. Use it as a checklist when updating Appendix H and the Results/Discussion text.

In [26]:
summary_rows = []

summary_rows.append(
    wild_bootstrap.assign(check="Wild cluster bootstrap")
    [["check", "model", "test", "p_cluster_t", "p_bootstrap"]]
)

print("Wild bootstrap vs cluster-t p-values:")
display(wild_bootstrap[["model", "test", "p_cluster_t", "p_bootstrap"]].round(4))

print("\nH3 diagnostics:")
display(h3_diagnostics[["model", "sample", "N", "F", "p"]].round(4))

print("\nIncremental R-squared:")
display(effect_sizes.round(3))

print("\nTables saved in:", TABLE_DIR)
print(sorted(p.name for p in TABLE_DIR.glob("*.csv")))

Wild bootstrap vs cluster-t p-values:


,model,test,p_cluster_t,p_bootstrap
0,Financial-RoBERTa,H1,0.0216,0.071
1,Financial-RoBERTa,H3,0.3161,0.569
2,FinBERT,H1,0.0000,0.004
3,FinBERT,H3,0.0025,0.047



H3 diagnostics:


,model,sample,N,F,p
0,Financial-RoBERTa,Without PERF,290,1.7630,0.1723
1,Financial-RoBERTa,Full sample,366,1.2599,0.3161
2,FinBERT,Without PERF,290,3.5574,0.0220
3,FinBERT,Full sample,366,5.2737,0.0025
4,Financial-RoBERTa,Excluding adidas,252,2.2348,0.1011
5,Financial-RoBERTa,Excluding puma,282,1.8121,0.1673
6,Financial-RoBERTa,Excluding asics,241,0.4591,0.8008
7,Financial-RoBERTa,Excluding mizuno,323,8.0280,0.0005
8,FinBERT,Excluding adidas,252,8.3335,0.0005
9,FinBERT,Excluding puma,282,2.3595,0.0874



Incremental R-squared:


,model,R2 additive,R2 valence only,dR2 context,dR2 valence,dR2 country,dR2 segment length,dR2 context x country,SD of outcome
0,Financial-RoBERTa,0.392,0.335,0.026,0.238,0.017,0.003,0.010,0.577
1,FinBERT,0.352,0.270,0.048,0.215,0.009,0.020,0.027,0.422



Tables saved in: /Users/Saowalak/Desktop/CEOCommunication/results/tables/improvements
['01_wild_cluster_bootstrap.csv', '02_alternative_outcome_models.csv', '03_cell_means_company_context.csv', '03_ordered_logit.csv', '04_h3_diagnostics.csv', '05_valence_context_country.csv', '06_valence_country_interaction.csv', '07_effect_sizes.csv', '08_other_included.csv', '09_adidas_interview_control.csv', '10_communicators.csv', '11_person_indicator.csv']
